In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
gemini_api_key = os.getenv("GEMINI_API_KEY")

if gemini_api_key is None:
    raise ValueError("GEMINI_API_KEY environment variable is not set.")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0, api_key=gemini_api_key)

chain_llm = llm | StrOutputParser()


In [ ]:

from typing import TypedDict

class AgentState(TypedDict):
    question: str
    model_name: str
    response: str

In [ ]:
def answer_question(state: AgentState) -> AgentState:

    question = state["question"]
    if not question:
        raise ValueError("Question is empty.")

    response = chain_llm.invoke(question)
    if response is None:
        raise ValueError("LLM failed to generate a response.")
    
    return {"response": response, "model_name": llm.model}

In [ ]:
from langgraph.graph import StateGraph

graph = StateGraph(AgentState)

graph.add_node("answer_question", answer_question)

In [ ]:
from langgraph.graph import START, END

graph.add_edge(START, "answer_question")
graph.add_edge("answer_question", END)

app = graph.compile()

In [ ]:
result = app.invoke({
    "question": "Tell me about virat kohili",
    "model_name": "",
    "response": "",
})

In [ ]:
print(result)